# 🧮 IA Calculadora — Investigación de Operaciones

Sistema inteligente que combina solvers numéricos (scipy, PuLP, Dijkstra) con RAG sobre Gemini para resolver problemas de IO.

**Arquitectura modular:**
- `config.py` — Configuración centralizada
- `embeddings.py` — ChromaDB + generación de embeddings
- `solvers.py` — Solucionadores numéricos
- `rag.py` — Consultas a Gemini con contexto del libro

In [1]:
# ============================================================
# 1. IMPORTS Y CARGA DEL SISTEMA
# ============================================================
import sys
import os

# Asegurar que los módulos se encuentren
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from config import GOOGLE_API_KEY
from embeddings import load_or_create_vectorstore
from solvers import resolver_lp_scipy, resolver_lp_pulp, agregar_variable_pulp, resolver_pulp, GrafoDijkstra
from rag import preguntar_io

print(f"✅ Módulos cargados")
print(f"✅ API Key configurada (termina en ...{GOOGLE_API_KEY[-4:]})")

INFO     | io_calc.main         | 📝 Log iniciado: d:\IA - Calculadora\src\..\logs\io_calc_20260415_211044.log
INFO     | io_calc.config       | API Key cargada (termina en ...HTyg)
INFO     | io_calc.config       | Modelos chat: ['gemini-2.5-flash', 'gemini-2.5-flash-lite']
INFO     | io_calc.config       | Modelo embeddings: models/gemini-embedding-001
INFO     | io_calc.config       | Configuración cargada correctamente


✅ Módulos cargados
✅ API Key configurada (termina en ...HTyg)


In [2]:
# ============================================================
# 2. CARGAR VECTOR STORE (ChromaDB)
# Primera vez: genera embeddings desde el PDF (~5 min)
# Siguientes veces: carga instantáneamente (0 API calls)
# ============================================================

vectorstore, embeddings_model = load_or_create_vectorstore()

INFO     | io_calc.embeddings   | === Iniciando carga/creación de vectorstore ===
INFO     | io_calc.embeddings   | Inicializando modelo de embeddings: models/gemini-embedding-001
INFO     | io_calc.embeddings   | Cargando PDF desde d:\IA - Calculadora\src\..\Insumos\Investigacion-Operaciones10Edicion-Frederick-S-Hillier.pdf
INFO     | io_calc.embeddings   | PDF cargado: 1229 páginas
INFO     | io_calc.embeddings   | 2996 fragmentos creados (chunk_size=2000, overlap=400)
INFO     | io_calc.embeddings   | Total de chunks deseados: 2996
INFO     | io_calc.embeddings   | ChromaDB encontrado en d:\IA - Calculadora\src\..\chroma_db
d:\IA - Calculadora\src\embeddings.py:150: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector

⚡ Vector store completo: 2996 fragmentos (0 API calls)


In [3]:
# ==================== EJEMPLOS DE USO ====================

print("\n" + "="*60)
print("EJEMPLO 1: Programación Lineal con scipy.optimize.linprog")
print("="*60)

# Problema: Maximizar 40x1 + 30x2
# Sujeto a: 2x1 + x2 <= 100 (horas máquina)
#           x1 + 2x2 <= 80 (horas mano de obra)
#           x1, x2 >= 0

c = [-40, -30]  # Negativo para maximizar (linprog minimiza)
A_ub = [[2, 1], [1, 2]]
b_ub = [100, 80]
bounds = [(0, None), (0, None)]

estado, solucion, valor, mensaje = resolver_lp_scipy(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds)

if estado == "ÓPTIMO":
    print(f"Solución óptima: x1 = {solucion[0]:.2f}, x2 = {solucion[1]:.2f}")
    print(f"Valor máximo de Z = {-valor:.2f}")
    print(f"Estado: {mensaje}")
else:
    print(f"No se encontró solución óptima: {mensaje}")


EJEMPLO 1: Programación Lineal con scipy.optimize.linprog
Solución óptima: x1 = 40.00, x2 = 20.00
Valor máximo de Z = 2200.00
Estado: Optimization terminated successfully. (HiGHS Status 7: Optimal)


In [4]:
print("\n" + "="*60)
print("EJEMPLO 2: Programación Lineal Entera con PuLP")
print("="*60)

prob = resolver_lp_pulp("maximizar")

x1 = agregar_variable_pulp(prob, "Producto_A", lowBound=0, cat='Integer')
x2 = agregar_variable_pulp(prob, "Producto_B", lowBound=0, cat='Integer')

prob += 40*x1 + 30*x2          # Función objetivo
prob += 2*x1 + x2 <= 100       # Horas máquina
prob += x1 + 2*x2 <= 80        # Horas mano de obra

estado, resultados, valor_objetivo = resolver_pulp(prob)

if estado == "ÓPTIMO":
    print("Solución óptima entera:")
    for var, val in resultados.items():
        print(f"  {var} = {val}")
    print(f"Valor máximo de Z = {valor_objetivo:.2f}")
else:
    print("No se encontró solución óptima")


EJEMPLO 2: Programación Lineal Entera con PuLP
Solución óptima entera:
  Producto_A = 40.0
  Producto_B = 20.0
Valor máximo de Z = 2200.00


In [5]:
print("\n" + "="*60)
print("EJEMPLO 3: Ruta más corta con Dijkstra")
print("="*60)

grafo = GrafoDijkstra()
grafo.agregar_arista_no_dirigida('A', 'B', 4)
grafo.agregar_arista_no_dirigida('A', 'C', 2)
grafo.agregar_arista_no_dirigida('B', 'C', 1)
grafo.agregar_arista_no_dirigida('B', 'D', 5)
grafo.agregar_arista_no_dirigida('C', 'D', 8)
grafo.agregar_arista_no_dirigida('C', 'E', 10)
grafo.agregar_arista_no_dirigida('D', 'E', 2)

print("Estructura del grafo:")
grafo.mostrar_grafo()

distancia, ruta = grafo.ruta_mas_corta('A', 'E')

if distancia is not None:
    print(f"\nRuta más corta de A a E:")
    print(f"  Ruta: {' -> '.join(ruta)}")
    print(f"  Distancia total: {distancia}")
else:
    print("No existe ruta entre los nodos")


EJEMPLO 3: Ruta más corta con Dijkstra
Estructura del grafo:
A -> {'B': 4, 'C': 2}
B -> {'A': 4, 'C': 1, 'D': 5}
C -> {'A': 2, 'B': 1, 'D': 8, 'E': 10}
D -> {'B': 5, 'C': 8, 'E': 2}
E -> {'C': 10, 'D': 2}

Ruta más corta de A a E:
  Ruta: A -> C -> B -> D -> E
  Distancia total: 10


In [6]:
print("\n" + "="*60)
print("EJEMPLO 4: Problema de Redes con Gemini (Flujo máximo)")
print("="*60)

pregunta_redes = """
Encontrar el flujo máximo desde el nodo origen S hasta el nodo destino T en la siguiente red:

S -> A: capacidad 10
S -> B: capacidad 5
A -> B: capacidad 15
A -> C: capacidad 10
B -> C: capacidad 5
B -> T: capacidad 10
C -> T: capacidad 10
"""

resultado = preguntar_io(pregunta_redes, vectorstore)
print(resultado)

INFO     | io_calc.rag          | ============================================================
INFO     | io_calc.rag          | NUEVA CONSULTA: 
Encontrar el flujo máximo desde el nodo origen S hasta el nodo destino T en la siguiente red:

S ->...
INFO     | io_calc.rag          | ============================================================
INFO     | io_calc.rag          | Cache HIT — clave: eca5ce7c0cf003759321c92cd2c1383d



EJEMPLO 4: Problema de Redes con Gemini (Flujo máximo)
⚡ Respuesta cargada desde caché (0 API calls)
¡Excelente! Este es un problema clásico de Investigación de Operaciones, y el contexto proporcionado es muy útil para guiar la resolución.

---

### 1. Clasificación del Problema

Este es un problema de **Redes**, específicamente un **Problema de Flujo Máximo**.

*   **Nodos:** S (Origen), A, B, C (Nodos de Transbordo), T (Destino).
*   **Aristas (Arcos) y Capacidades:**
    *   S -> A: capacidad 10
    *   S -> B: capacidad 5
    *   A -> B: capacidad 15
    *   A -> C: capacidad 10
    *   B -> C: capacidad 5
    *   B -> T: capacidad 10
    *   C -> T: capacidad 10

---

### 2. Formulación Matemática Completa

El problema de flujo máximo se puede formular como un programa lineal.

**Variables de Decisión:**
Sea $x_{ij}$ la cantidad de flujo que pasa por el arco dirigido desde el nodo $i$ al nodo $j$.

**Función Objetivo:**
Maximizar el flujo total desde el origen (S) hasta el destin

In [7]:
print("\n" + "="*60)
print("EJEMPLO 5: PERT/CPM con datos del problema")
print("="*60)

pregunta_pert = """
Un proyecto tiene las siguientes actividades:
A: duración 3 días, predecesores: ninguna
B: duración 5 días, predecesores: A
C: duración 2 días, predecesores: A
D: duración 4 días, predecesores: B, C

Calcular:
- Tiempo early y late de cada evento
- Holguras
- Ruta crítica
"""

resultado2 = preguntar_io(pregunta_pert, vectorstore)
print(resultado2)

INFO     | io_calc.rag          | ============================================================
INFO     | io_calc.rag          | NUEVA CONSULTA: 
Un proyecto tiene las siguientes actividades:
A: duración 3 días, predecesores: ninguna
B: duración...
INFO     | io_calc.rag          | ============================================================
INFO     | io_calc.rag          | Buscando contexto relevante en ChromaDB...



EJEMPLO 5: PERT/CPM con datos del problema
🔍 Buscando contexto relevante...


INFO     | io_calc.embeddings   | Contexto encontrado — Páginas: [986, 990, 959, 962, 987], Scores: ['0.445', '0.457', '0.470', '0.471', '0.472']
INFO     | io_calc.rag          | --- Intentando modelo 1/2: gemini-2.5-flash ---


  📖 Contexto extraído de páginas: [986, 990, 959, 962, 987]
  📊 Scores de distancia: ['0.445', '0.457', '0.470', '0.471', '0.472']


INFO     | io_calc.rag          | Enviando prompt a gemini-2.5-flash (intento 1/3)...


🤖 Consultando gemini-2.5-flash...


INFO     | io_calc.rag          | ✅ Respuesta recibida de gemini-2.5-flash en 35.9s (9972 chars)
INFO     | io_calc.rag          | Respuesta guardada en caché (clave: 149e30045fbc123dead1d7036540aaf7)


💾 Respuesta guardada en caché
¡Excelente! Este es un problema clásico de **Gestión de Proyectos** utilizando el **Método de la Ruta Crítica (CPM)**, que es una técnica dentro del marco de **PERT/CPM**. Se clasifica como un problema de **Redes**.

---

### 1. Clasificación del Problema

El problema es de **Redes de Proyectos**, específicamente utilizando el **Método de la Ruta Crítica (CPM)**. Se trata de determinar la secuencia de actividades que define la duración mínima del proyecto, así como los tiempos de inicio y finalización más tempranos y más tardíos para cada actividad y evento, y la holgura asociada.

---

### 2. Formulación Matemática (Conceptual para CPM)

El CPM no se formula típicamente como un problema de programación lineal con una función objetivo y restricciones explícitas en el sentido tradicional de un LP, sino como un algoritmo basado en cálculos de red. Sin embargo, podemos describir los componentes y las relaciones matemáticas:

**Objetivo:** Minimizar la duració

---
## 🛠️ Utilidades

In [11]:
# ============================================================
# CONSULTA LIBRE: Escribe tu propio problema de IO
# ============================================================

mi_pregunta = """
Una empresa de muebles fabrica mesas y sillas. Cada mesa requiere 4 horas de carpintería 
y 2 horas de acabado. Cada silla requiere 3 horas de carpintería y 1 hora de acabado. 
Se dispone de 240 horas de carpintería y 100 horas de acabado por semana. 
La ganancia por mesa es de $70 y por silla es de $50.
Además, se deben producir al menos 10 sillas por semana por contrato con un distribuidor, 
y no se pueden producir más de 40 mesas por limitaciones de almacén.
Formular y resolver el problema de programación lineal para maximizar las ganancias semanales.
¿Cuál es la solución óptima, cuáles restricciones son activas, y cuál es la solucion para la maxima ganancia?
"""

resultado = preguntar_io(mi_pregunta, vectorstore)
print(resultado)

INFO     | io_calc.rag          | ============================================================
INFO     | io_calc.rag          | NUEVA CONSULTA: 
Una empresa de muebles fabrica mesas y sillas. Cada mesa requiere 4 horas de carpintería 
y 2 horas...
INFO     | io_calc.rag          | ============================================================
INFO     | io_calc.rag          | Buscando contexto relevante en ChromaDB...


🔍 Buscando contexto relevante...


INFO     | io_calc.embeddings   | Contexto encontrado — Páginas: [290, 107, 106, 931, 289], Scores: ['0.411', '0.443', '0.444', '0.453', '0.457']
INFO     | io_calc.rag          | --- Intentando modelo 1/2: gemini-2.5-flash ---


  📖 Contexto extraído de páginas: [290, 107, 106, 931, 289]
  📊 Scores de distancia: ['0.411', '0.443', '0.444', '0.453', '0.457']


INFO     | io_calc.rag          | Enviando prompt a gemini-2.5-flash (intento 1/3)...


🤖 Consultando gemini-2.5-flash...


INFO     | io_calc.rag          | ✅ Respuesta recibida de gemini-2.5-flash en 22.7s (6033 chars)
INFO     | io_calc.rag          | Respuesta guardada en caché (clave: f6f10f458a466ba3de62abfe90830cb4)


💾 Respuesta guardada en caché
¡Excelente! Como experto en Investigación de Operaciones, analizaré este problema de Programación Lineal paso a paso.

---

### 1. Clasificación del Problema

Este es un problema de **Programación Lineal (LP)**. Se busca optimizar (maximizar) una función objetivo lineal (ganancia) sujeta a un conjunto de restricciones lineales (recursos disponibles, requisitos de producción, límites de capacidad).

---

### 2. Formulación Matemática Completa

**Variables de Decisión:**
*   `x1`: Número de mesas a producir por semana.
*   `x2`: Número de sillas a producir por semana.

**Función Objetivo (Maximizar Ganancia):**
*   Cada mesa genera $70 de ganancia.
*   Cada silla genera $50 de ganancia.
*   `Maximizar Z = 70x1 + 50x2`

**Restricciones:**

1.  **Horas de Carpintería:**
    *   Cada mesa requiere 4 horas.
    *   Cada silla requiere 3 horas.
    *   Disponibilidad: 240 horas por semana.
    *   `4x1 + 3x2 <= 240`

2.  **Horas de Acabado:**
    *   Cada mesa re

In [9]:
# ============================================================
# UTILIDAD: Regenerar embeddings o limpiar cachés
# ============================================================

# Descomenta la acción que necesites:

# --- Regenerar ChromaDB (volver a procesar el PDF) ---
# from embeddings import delete_vectorstore
# delete_vectorstore()

# --- Limpiar caché de respuestas ---
# from rag import limpiar_cache_respuestas
# limpiar_cache_respuestas()